In [1]:
# %pip install pandas
# %pip install matplotlib
# %pip install scikit-learn
# %pip install xgboost
# %pip install numpy

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import joblib

In [3]:
print(os.getcwd())
project_root = Path.cwd().parent

/home/chiyedza/Downloads/Personal projects/SA-Water-Dam-Level-Predictor/notebooks


In [4]:
"""Feature_engineered file path"""
featured_data_csv = "../data/feature_engineered/engineered_data.csv"

In [5]:
df = pd.read_csv(featured_data_csv)
df = df.sort_values("date").reset_index(drop=True)
df["date"] = pd.to_datetime(df["date"])

In [6]:
# df = df.drop(["lag_1_target_30"], axis=1)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4659 entries, 0 to 4658
Data columns (total 26 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         4659 non-null   datetime64[us]
 1   water_level_mm               4659 non-null   float64       
 2   year                         4659 non-null   int64         
 3   month                        4659 non-null   int64         
 4   lag_1                        4659 non-null   float64       
 5   lag_7                        4659 non-null   float64       
 6   lag_14                       4659 non-null   float64       
 7   lag_28                       4659 non-null   float64       
 8   lag_56                       4659 non-null   float64       
 9   lag_84                       4659 non-null   float64       
 10  rolling_mean_28              4659 non-null   float64       
 11  rolling_mean_84              4659 non-null   float64  

In [7]:
features = ["lag_1", "lag_7", "lag_14", "lag_28", "lag_56", "lag_84",
            "rolling_mean_28", "rolling_mean_84",
            "lag_7_weekly_sum", "weekly_sum", "week_over_week_pct",
            "significant_rain_days", "days_since_significant_rain",
            "quarter", "wet_season", "days_into_season", "season_start_year"
            ]
targets = ["target_30", "target_60", "target_90"]

In [8]:
for target in targets:

    target_name = target
    X = df[features].values
    y = df[target].values
    print(target_name)
    
    n_splits = 5
    gap_config = {
        "target_30": 30,
        "target_60": 60,
        "target_90":90
    }

    gap = gap_config[target_name]

    n = len(df)
    fold_size = n//(n_splits + 1)

    scores = []
    # count = 0
    # for target in targets:
    #     count += 1
    for fold in range(1, n_splits + 1):
        train_end = fold * fold_size
        test_start = train_end + gap
        test_end = test_start + fold_size

        if test_end > n:
            break
        
        X_train = X[:train_end]
        y_train = y[:train_end]
        X_test = X[test_start: test_end]
        y_test = y[test_start: test_end]

        model = XGBRegressor(
            n_estimators = 500,
            learning_rate = 0.05,
            max_depth = 6,
            random_state = 0
        )
        
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        mae = mean_absolute_error(y_test, preds)
        scores.append(mae)

        train_dates = f"{df.date.iloc[0].year}-{df.date.iloc[train_end-1].year}"
        test_dates = f"{df.date.iloc[test_start].year}-{df.date.iloc[test_end-1].year}"

        
        print(f"Fold {fold}: Train {train_dates} -> Test {test_dates}")
        print(f"              MAE: {mae}")
        print( )
        

        avg_mae = np.mean(scores)
        std_mae = np.std(scores)
        trend = scores[-1] - scores[0]
        baseline_mae = 23

        print(f"Average MAE:   {avg_mae}")
        print(f"Standard deviation: {std_mae}")
        print(f"Trend (last-first)   {trend:+.2f}")
        print(f"Improvement vs mean: {((baseline_mae - avg_mae) / baseline_mae)}")
    scores.clear()

target_30
Fold 1: Train 2000-2004 -> Test 2004-2009
              MAE: 14.982496010538938

Average MAE:   14.982496010538938
Standard deviation: 0.0
Trend (last-first)   +0.00
Improvement vs mean: 0.34858712997656793
Fold 2: Train 2000-2009 -> Test 2009-2013
              MAE: 12.039621017707319

Average MAE:   13.511058514123128
Standard deviation: 1.4714374964158097
Trend (last-first)   -2.94
Improvement vs mean: 0.41256267329899443
Fold 3: Train 2000-2013 -> Test 2013-2017
              MAE: 14.401433413838312

Average MAE:   13.807850147361522
Standard deviation: 1.2726309039516963
Trend (last-first)   -0.58
Improvement vs mean: 0.3996586892451512
Fold 4: Train 2000-2017 -> Test 2017-2022
              MAE: 16.83187124915457

Average MAE:   14.563855422809784
Standard deviation: 1.7115268016559047
Trend (last-first)   +1.85
Improvement vs mean: 0.3667888946604442
target_60
Fold 1: Train 2000-2004 -> Test 2005-2009
              MAE: 23.404273287816313

Average MAE:   23.40427328781